# Dreamer V1 PyTorch Cartpole Swingup Pilot

This notebook stops the TensorFlow route and reuses Kaggle's existing PyTorch GPU environment. It retrieves `juliusfrost/dreamer-pytorch` at commit `ccea6ae4a397a94c328891bd78e81d52dd156cb6` with the pinned `rlpyt` submodule, without reinstalling PyTorch.

The first cell runs a bounded gate: 64x64 RGB pixels, camera 0, action repeat 2, episode reset contract, ten real Dreamer updates, finite diagnostics, parameter changes for world model/actor/critic, checkpoint save, and fresh-process same-seed evaluation. Only after that gate passes should the bounded single-seed baseline cell be run. README scores are not our results.


In [ ]:
import base64
from pathlib import Path

root = Path('/kaggle/working/dreamer_pytorch_pilot')
root.mkdir(parents=True, exist_ok=True)
runner = root / 'kaggle_pytorch_pilot.py'
runner.write_bytes(base64.b64decode('...'))
print(runner)


## Preflight and ten-update gate

The default Kaggle interpreter is used only to launch the subprocess. The actual Dreamer code runs in that same existing PyTorch environment, with `CUDA_VISIBLE_DEVICES=0` set before framework import.


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, str(runner), '--phase', 'debug'], check=True)
print('PyTorch Dreamer gate passed; the baseline cell is now authorized.')


## Bounded single-seed cartpole_swingup baseline

This is a pilot budget, not a complete 1M/5M-step paper reproduction. Adjust `--n-steps` only within the available Kaggle session budget; do not run this cell if the gate failed.


In [ ]:
subprocess.run([sys.executable, str(runner), '--phase', 'train', '--n-steps', '10000'], check=True)


Outputs are written to `/kaggle/working/dreamer_pytorch_pilot/`, including `source_manifest.json`, `compatibility_patches.json`, `metrics.jsonl`, `debug_summary.json`, `train_summary.json`, `eval_episodes.csv`, `summary.json`, `progress.csv`, and the rlpyt checkpoint.
